# Phase 11b Pilot — ALFWorld (Embodied Household Navigation)

**Goal**: Verify that ALFWorld runs on Colab A100 and that spectral entropy features from an
agent's per-step generations can be extracted and linked to task outcome (success/failure).
This is a **GO/NO-GO pilot** — N=5 tasks, one model.

## Gates
| ID | Gate | Threshold | Required? |
|----|------|-----------|-----------|
| G0 | `import alfworld` succeeds without corrupting numpy/pyarrow | Clean import | **Required** |
| G1 | `env.step()` returns a valid observation string | No crash | **Required** |
| G2 | At least 1 trajectory reaches `task_success = True` | ≥ 1/5 | Informative |
| G3 | Entropy traces non-degenerate (std > 0) | ≥ 80% of steps | Informative |
| G4 | `extract_all_features` returns valid dict | ≥ 70% of steps | Informative |

**Verdict**: G0 + G1 required (env must work); G2–G4 informative → GO for Phase 11b full run.

In [ ]:
# ── Cell 2: Setup (spectral_utils only — alfworld deferred to Cell 3) ─────────
import os, sys, shutil

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/hallucination_detection'
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes autoawq scipy')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, sw_var_peak_adaptive, FEAT_NAMES,
    load_cache, save_cache,
    zscore, boot_auc, nadler_fuse, best_nadler_on,
)
import datasets  # freeze pyarrow before gptqmodel install
print('spectral_utils imported OK')
print(f'FEAT_NAMES ({len(FEAT_NAMES)}):', FEAT_NAMES)

In [ ]:
# ── Cell 3: Install ALFWorld + verify import (G0 gate) ───────────────────────
# ALFWorld requires a virtual display on headless Colab.
# Install separately from Cell 2 to isolate any numpy/pyarrow corruption risk.
import subprocess

ret = subprocess.run(
    ['pip', 'install', '-q', 'alfworld', 'pyvirtualdisplay'],
    capture_output=True, text=True
)
print(ret.stdout[-500:] if ret.stdout else '(no stdout)')
print(ret.stderr[-500:] if ret.stderr else '')

# Virtual display for ALFWorld's renderer
os.system('apt-get install -q -y xvfb')
from pyvirtualdisplay import Display
_display = Display(visible=0, size=(1024, 768))
_display.start()
print('Virtual display started.')

# G0: verify alfworld imports cleanly
try:
    import alfworld
    import alfworld.agents
    import numpy as _np_check
    import pyarrow as _pa_check
    print(f'G0 PASS: alfworld {alfworld.__version__} imported; numpy={_np_check.__version__}, pyarrow={_pa_check.__version__}')
    G0_PASS = True
except Exception as e:
    print(f'G0 FAIL: {e}')
    G0_PASS = False

In [ ]:
# ── Cell 4: Config ────────────────────────────────────────────────────────────
MODEL_KEY    = 'qwen25_7b'
MODEL_ID     = 'Qwen/Qwen2.5-7B-Instruct'
N_TASKS      = 5
MAX_STEPS    = 20
MAX_NEW      = 64
TEMP         = 1.0
TASK_FILTER  = 'pick_and_place'   # smallest ALFWorld task type

CACHE_DIR = '/content/drive/MyDrive/hallucination_detection/cache/phase11b_alfworld_pilot'
RAW_DIR   = os.path.join(CACHE_DIR, 'raw')
FEAT_DIR  = os.path.join(CACHE_DIR, 'features')
RES_DIR   = os.path.join(CACHE_DIR, 'results')
for d in [RAW_DIR, FEAT_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

CELL_KEY = f'{MODEL_KEY}_n{N_TASKS}'
print(f'Config: {MODEL_KEY} | N={N_TASKS} tasks | max_steps={MAX_STEPS} | task_filter={TASK_FILTER!r}')
print(f'Cache: {CACHE_DIR}')

In [ ]:
# ── Cell 5: Env smoke test (G1 gate) ─────────────────────────────────────────
# Load env, send one hardcoded action, verify we get an observation back.
assert G0_PASS, "G0 failed — fix alfworld install before proceeding"

from spectral_utils.alfworld_utils import (
    setup_alfworld_env, alfworld_action_prompt, parse_alfworld_action, run_alfworld_episode,
)

env, task_descs = setup_alfworld_env(task_filter=TASK_FILTER)
print(f'\nAvailable tasks (first 10):')
for td in task_descs[:10]:
    print(f'  {td}')

# Smoke test: one step with first task
obs_list, infos = env.reset()
obs = obs_list[0] if isinstance(obs_list, list) else obs_list
admissible = list(infos.get('admissible_commands', [[]])[0])
print(f'\n--- Smoke test ---')
print(f'Observation: {obs[:200]}')
print(f'Admissible actions ({len(admissible)}): {admissible[:5]}')

# Send first admissible action
action = admissible[0] if admissible else 'look'
obs2_list, _, done_list, infos2 = env.step([action])
obs2 = obs2_list[0] if isinstance(obs2_list, list) else obs2_list

G1_PASS = isinstance(obs2, str) and len(obs2) > 0
print(f'\nAction sent: {action!r}')
print(f'Response: {obs2[:200]}')
print(f'G1 {"PASS" if G1_PASS else "FAIL"}: env.step returned valid observation string')

In [ ]:
# ── Cell 6: Load model ────────────────────────────────────────────────────────
assert G1_PASS, "G1 failed — env.step broken, stop here"
mdl, tok = load_model(MODEL_ID)
print('Model loaded.')

In [ ]:
# ── Cell 7: Inference ─────────────────────────────────────────────────────────
import pickle, time

RAW_PATH    = os.path.join(RAW_DIR, f'{CELL_KEY}_trajs.pkl')
FORCE_RERUN = False

if not FORCE_RERUN and os.path.exists(RAW_PATH):
    with open(RAW_PATH, 'rb') as f:
        trajs = pickle.load(f)
    print(f'Loaded {len(trajs)} trajectories from {RAW_PATH}')
else:
    trajs = []
    tasks_to_run = task_descs[:N_TASKS]
    t0 = time.time()

    for i, td in enumerate(tasks_to_run):
        # Reset env to move to next game (ALFWorld cycles games on reset)
        env.reset()
        traj = run_alfworld_episode(
            mdl, tok, env, task_desc=td,
            T=TEMP, max_steps=MAX_STEPS, max_new=MAX_NEW,
        )
        traj['task_idx'] = i
        trajs.append(traj)

        elapsed = time.time() - t0
        print(f'[{i+1}/{N_TASKS}] {td[:50]:50s} | success={traj["task_success"]} | '
              f'steps={traj["n_steps"]} | {elapsed:.0f}s elapsed')

        # Checkpoint every task (N=5 is small — checkpoint every 1)
        with open(RAW_PATH, 'wb') as f:
            pickle.dump(trajs, f)

    print(f'\nDone. Saved {len(trajs)} trajs to {RAW_PATH}')

n_success = sum(t['task_success'] for t in trajs)
print(f'Task success: {n_success}/{len(trajs)}')

In [ ]:
# ── Cell 8: Unload model ──────────────────────────────────────────────────────
del mdl, tok
free_memory()
print('Model unloaded.')

In [ ]:
# ── Cell 9: Feature extraction ────────────────────────────────────────────────
# Flatten trajectory steps: each step is one sample.
# Label: task_success (same label for all steps of a trajectory).
import pickle, numpy as np
from spectral_utils import extract_all_features, sw_var_peak_adaptive

FEAT_PATH = os.path.join(FEAT_DIR, f'{CELL_KEY}_feats.pkl')
FORCE_RECOMPUTE = False

if not FORCE_RECOMPUTE and os.path.exists(FEAT_PATH):
    with open(FEAT_PATH, 'rb') as f:
        feat_data = pickle.load(f)
    print(f'Loaded features from {FEAT_PATH}')
else:
    feat_data = {'features': [], 'labels': [], 'step_idx': [], 'task_idx': []}
    n_nan = 0

    for traj in trajs:
        label = int(traj['task_success'])
        for step_i, step in enumerate(traj['steps']):
            ents = step['token_entropies']
            if len(ents) < 4:
                n_nan += 1
                continue
            feats = extract_all_features(ents)
            feats['sw_var_peak'] = sw_var_peak_adaptive(ents)
            feat_data['features'].append(feats)
            feat_data['labels'].append(label)
            feat_data['step_idx'].append(step_i)
            feat_data['task_idx'].append(traj['task_idx'])

    with open(FEAT_PATH, 'wb') as f:
        pickle.dump(feat_data, f)
    print(f'Extracted {len(feat_data["labels"])} step features ({n_nan} too-short skipped)')

labels = np.array(feat_data['labels'])
n_pos, n_neg = labels.sum(), len(labels) - labels.sum()
print(f'Label balance: {n_pos} success / {n_neg} failure steps')

# G3 check: entropy degeneracy
all_steps = [s for t in trajs for s in t['steps'] if len(s['token_entropies']) >= 4]
non_degen = sum(1 for s in all_steps if np.std(s['token_entropies']) > 0)
print(f'G3 check: {non_degen}/{len(all_steps)} steps have std > 0')

In [ ]:
# ── Cell 10: GO/NO-GO Gate Cell ───────────────────────────────────────────────
import numpy as np

n_total_steps = len(all_steps)
n_valid_feats = len(feat_data['labels'])
n_success_tasks = sum(t['task_success'] for t in trajs)

g0 = G0_PASS
g1 = G1_PASS
g2 = n_success_tasks >= 1
g3 = non_degen / max(n_total_steps, 1) >= 0.80
g4 = n_valid_feats / max(n_total_steps, 1) >= 0.70

print("=" * 55)
print("  Phase 11b ALFWorld Pilot — GO/NO-GO Report")
print("=" * 55)
print(f"  G0  alfworld imports cleanly:     {'PASS ✓' if g0 else 'FAIL ✗'}  [REQUIRED]")
print(f"  G1  env.step works:               {'PASS ✓' if g1 else 'FAIL ✗'}  [REQUIRED]")
print(f"  G2  ≥1 task solved:               {'PASS ✓' if g2 else 'FAIL ✗'}  ({n_success_tasks}/{len(trajs)}) [informative]")
print(f"  G3  Non-degen traces ≥80%:        {'PASS ✓' if g3 else 'FAIL ✗'}  ({non_degen}/{n_total_steps}) [informative]")
print(f"  G4  Feature coverage ≥70%:        {'PASS ✓' if g4 else 'FAIL ✗'}  ({n_valid_feats}/{n_total_steps}) [informative]")
print("=" * 55)

required_pass = g0 and g1
verdict = "GO  → proceed to Phase 11b full ALFWorld run" if required_pass \
          else "NO-GO → G0 or G1 failed, fix environment before proceeding"
print(f"  VERDICT: {verdict}")
print("=" * 55)

In [ ]:
# ── Cell 11: Indicative AUC (N=5 tasks, step-level) ──────────────────────────
# With only 5 tasks the AUC is purely indicative. Focus is on feature plumbing.
import numpy as np
from spectral_utils import boot_auc, FEAT_NAMES

feats_list = feat_data['features']
labels_arr = np.array(feat_data['labels'])

if len(np.unique(labels_arr)) < 2:
    print('Only one class present (all tasks same outcome) — AUC not meaningful.')
    print('This is expected for N=5 pilot. Check GO/NO-GO verdict and proceed.')
else:
    rows = []
    for feat_name in FEAT_NAMES + ['sw_var_peak']:
        vals = np.array([f.get(feat_name, float('nan')) for f in feats_list])
        mask = ~np.isnan(vals)
        if mask.sum() < 4 or len(np.unique(labels_arr[mask])) < 2:
            continue
        auc, lo, hi = boot_auc(vals[mask], labels_arr[mask])
        if auc < 0.5:
            auc, lo, hi = 1 - auc, 1 - hi, 1 - lo
        rows.append((feat_name, auc, lo, hi))

    rows.sort(key=lambda x: -x[1])
    print(f"Step-level AUC (N={len(labels_arr)} steps, {len(trajs)} tasks — indicative only)")
    print(f"\n{'Feature':<28} {'AUC':>6}  {'95% CI':>13}")
    print("-" * 52)
    for feat_name, auc, lo, hi in rows[:10]:
        print(f"  {feat_name:<26} {auc:.3f}  [{lo:.3f}, {hi:.3f}]")